# HCI2000 Data Preprocessing and Analysis



# 0.0 Setup


## 0.1 Explanation of raw data

The study produces **four raw data sources**, listed below in the order they are collected during a session:

| # | File | When it is collected | One row = | Rows (16 participants) |
|---|---|---|---|---|
| 1 | `screening_survey.csv` | Before the session | one person who applied | 18 (16 enrolled + 2 screened out) |
| 2 | `main_data.csv` (×16) | During the 24 retrieval trials | one trial | 16 files × 24 = **384** |
| 3 | `block_survey.csv` | After each of the 4 condition blocks | one participant × one condition | 16 × 4 = **64** |
| 4 | `weighting_form.csv` | Once, after all 4 blocks (before the interview) | one participant | **16** |

**The participant code (`P001`–`P016`) is the key that links all four files.** The controller records it automatically. In the Google Forms it is typed in by the researcher (`Participant Code (Filled by Researcher)`).

The three survey files are Google Forms exports, so their column names are the full question text. We will rename them to short names during cleaning.

### 1. `screening_survey.csv`: screening survey

Answered by everyone who signs up, **before** the session. It checks eligibility (consent, regular laptop/desktop browsing) and describes the sample (age, Chrome use, tab habits, usual tab orientation).

- **18 rows × 11 columns**: 16 enrolled participants + 2 people who were screened out. One declined consent, and one does not regularly browse on a laptop or desktop.
- The last column, `Participant Code (Filled by Researcher)`, is filled in by hand after the person completes the session. **It is blank for anyone who was screened out**, which is how we filter them.
- It contains participants' **names**. These are only needed to match people to codes. Drop the name column before analysis or sharing.
- Used for: describing the participants (e.g. age, how many use Chrome), not as an outcome.

### 2. `main_data.csv`: trial performance (written by the controller)

The Hammerspoon controller (`controller.lua`) writes this file automatically while the participant does the retrieval trials. It is the **main analysis table** for retrieval time and accuracy.

**Where it is:** each session gets its own folder, `records/main/<participant>/<session_id>/main_data.csv`. That gives **16 separate files**, one per participant, and the first preprocessing step is to **stack them into one table of 384 rows**. The other files in each folder (`events.csv`, `trials_aux.csv`, `session.json`, `checkpoint.json`) are the controller's detailed logs and backups. We don't need them for the analysis.

**Size:** each file has **24 rows** (4 conditions × 6 trials) and **25 columns**.

The 4 conditions combine **tab orientation** (H = horizontal, V = vertical) with **tab load** (L = low, 5 task tabs; H = high, 15 task tabs). Each participant does them in a counterbalanced order set by their `group`.

| Column group | Columns | Meaning |
|---|---|---|
| Who / which session | `session_id`, `mode`, `participant`, `group` | `mode` is `main` for real data (`test`/`pilot` otherwise). `group` (1–4) sets the condition order. |
| Which condition | `block`, `condition`, `orientation`, `tab_load`, `content_set` | `block` = order done (1–4). `condition` = `H-L`, `H-H`, `V-L`, `V-H`. `content_set` = topic set A–D. |
| Which trial | `trial`, `target_id`, `target_title`, `is_repeat`, `target_task_position`, `task_tab_count`, `visible_tab_count` | `trial` = T1–T6. `is_repeat` = 1 if this target already appeared in the block. `target_task_position` = where the target tab sits (1 = first). Tab counts: 5/15 task tabs, 6/16 visible (plus *Study Start*). |
| How the trial ended | `controller_status`, `measurement_source` | See statuses below. |
| **Outcomes** | `retrieval_time_s`, `total_tab_switches`, `incorrect_selections`, `first_selection_correct` | Time from tabs being revealed to the target being opened. Number of tabs opened. Number of wrong tabs. Whether the first tab opened was the target. |
| Problem flags | `no_selection`, `technical_loss`, `notes` | `no_selection` = timed out without opening any tab. `technical_loss` = interrupted by a technical problem. `notes` is always blank. |

**`controller_status`, how each trial ended:**
- `completed_auto`: the controller saw the target tab open. **Only these rows have a `retrieval_time_s`.**
- `completed_manual`: the facilitator marked it complete by hand. No time is recorded.
- `timeout`: the target was not reached within **20 seconds**. No time is recorded.
- `interrupted`: the trial was stopped, either by a technical problem (`technical_loss = true`) or by a person, e.g. a participant break (`technical_loss = false`). Interrupted trials are not repeated.

For a completed trial, `total_tab_switches = incorrect_selections + 1` (the wrong tabs plus the final correct one).

### 3. `block_survey.csv`: NASA-TLX ratings after each block

Participants fill in this Google Form **after each of the 4 condition blocks**. It measures how demanding that condition felt.

- **64 rows × 10 columns**: 16 participants × 4 conditions. Columns: timestamp, participant code, condition, the six ratings, and an optional comment.
- `What condition block did you just finish?` holds labels like `H-L: Horizontal Low`. We shorten these to `H-L` so they match `condition` in `main_data.csv`. **The two files are joined on participant + condition**, never on block order or row order.
- Six workload ratings, each from **0 to 10**: Mental Demand, Physical Demand, Temporal Demand, Performance, Effort, Frustration. We multiply them by 10 to get the standard **0–100** NASA-TLX scale.
- Performance is the one reversed item: a low rating means the participant felt they did *well*.
- The last column is an optional free-text comment.
- On its own this file gives the **raw (unweighted) TLX**: the average of the six ratings. To get the weighted score, it is combined with the weighting form below.

### 4. `weighting_form.csv`: NASA-TLX pairwise weights

Answered **once per participant, after all 4 blocks and before the interview**. The participant sees all **15 possible pairs** of the six workload dimensions and, for each pair, picks the one that contributed more to their workload.

- **16 rows × 17 columns**: timestamp, participant code, and 15 pair columns.
- Each pair cell contains the **name of the dimension that was picked** (e.g. `Mental Demand`).
- A dimension's **weight** is how many times it was picked, from 0 to 5. The six weights always add up to 15.
- The weights belong to the participant, not a condition, so the same weights are applied to all four of that participant's block ratings:

$$\text{Weighted TLX} = \frac{\sum_{d=1}^{6} (\text{rating}_d \times 10) \times \text{weight}_d}{15} \quad (0\text{–}100)$$

- All 15 pair headers use the same format, `<Dimension> vs. <Dimension>`. Even so, we count the dimension names in the **cells**, not the headers, because that is what actually records each choice.

## 1.0 Data Cleaning and Preprocessing

# 2.0 Data Analysis